# 3 · Pull: negotiate a contract, then fetch the robots

The consumer and the provider negotiate a contract over the dataset step by step (DSP contract negotiation), then the consumer opens a pull transfer on that contract and fetches the robots through the provider's dataplane.

In [ ]:
%run ./02_catalog.ipynb

In [ ]:
steps = []   # every DSP step of this notebook, for the timeline at the end


def rpc(base, path, payload, title=None):
    """POST one DSP step to an agent, show the call and keep it for the timeline."""
    response = requests.post(base + path, headers=json_header, data=json.dumps(payload))
    body = show(response, title)
    steps.append({
        "agent": helpers.AGENTS.get(base, base),
        "step": path.split("/dsp/current/", 1)[-1],
        "state": str(helpers.state_of(body) or "").rsplit(":", 1)[-1],
        "ms": int(response.elapsed.total_seconds() * 1000),
    })
    return body


def timeline(title):
    table(steps, [("Agent", "agent"), ("Step", "step"), ("State", "state"), ("ms", "ms")], title=title)


# The offer goes back exactly as the catalog advertises it.
offer = {
    "@id": policy_id,
    "@type": "Offer",
    "target": target_id,
    "permission": policy.get("permission") or [],
    "obligation": policy.get("obligation") or [],
    "prohibition": policy.get("prohibition") or [],
}

# Contract negotiation

```
REQUESTED → OFFERED → REQUESTED → OFFERED → ACCEPTED → AGREED → VERIFIED → FINALIZED
```

Each step is a call to the agent that sends the message: it signs it and delivers it to the other agent over DSP.

## The consumer requests the offer (consumer → provider)

In [ ]:
payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "offer": offer,
}
body = rpc(consumer, "/dsp/current/negotiations/rpc/setup-request-init", payload, "1 · Contract request")
pids = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}

## The provider makes an offer (provider → consumer)

In a real negotiation the terms can change at each step; here they stay as published.

In [ ]:
rpc(provider, "/dsp/current/negotiations/rpc/setup-offer", {**pids, "offer": offer}, "2 · Offer");

## The consumer requests again, on that offer (consumer → provider)

In [ ]:
rpc(consumer, "/dsp/current/negotiations/rpc/setup-request", {**pids, "offer": offer}, "3 · Request on the offer");

## The provider confirms its final terms (provider → consumer)

In [ ]:
rpc(provider, "/dsp/current/negotiations/rpc/setup-offer", {**pids, "offer": offer}, "4 · Final offer");

## The consumer accepts (consumer → provider)

In [ ]:
rpc(consumer, "/dsp/current/negotiations/rpc/setup-acceptance", pids, "5 · Acceptance");

## The provider agrees (provider → consumer)

The agreement is created here, signed by the provider.

In [ ]:
rpc(provider, "/dsp/current/negotiations/rpc/setup-agreement", pids, "6 · Agreement");

## The consumer verifies the agreement (consumer → provider)

In [ ]:
rpc(consumer, "/dsp/current/negotiations/rpc/setup-verification", pids, "7 · Verification");

## The provider finalizes (provider → consumer)

In [ ]:
body = rpc(provider, "/dsp/current/negotiations/rpc/setup-finalization", pids, "8 · Finalization")
agreement = body["negotiationAgentModel"]["agreement"]
agreement_id = agreement["id"]

## The agreement

Both agents keep it; every transfer refers to it.

In [ ]:
table(
    [
        {"field": "Agreement", "value": agreement_id},
        {"field": "Target", "value": agreement.get("target") or target_id},
        {"field": "Consumer", "value": short(agreement.get("consumerParticipantId", ""), 50)},
        {"field": "Provider", "value": short(agreement.get("providerParticipantId", ""), 50)},
        {"field": "Created", "value": agreement.get("createdAt", "")},
    ],
    [("", "field"), ("", "value")],
    title="Agreement",
)
timeline("The negotiation, step by step");

# Pull transfer

## The consumer requests a transfer on the agreement (consumer → provider)

In [ ]:
payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "agreementId": agreement_id,
    "format": "HTTP_PULL",
}
body = rpc(consumer, "/dsp/current/transfers/rpc/setup-request", payload, "Transfer request")
transfer = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}

## The provider starts it (provider → consumer)

The start message carries the data address: a dataplane URL on the provider, valid for this transfer only.

In [ ]:
from urllib.parse import urlsplit, urlunsplit

body = rpc(provider, "/dsp/current/transfers/rpc/setup-start", transfer, "Transfer start")

# The agent writes its container address; from here the provider is at its own URL.
parts, here = urlsplit(helpers.find_dataplane(body)), urlsplit(provider)
dataplane = urlunsplit((here.scheme, here.netloc, parts.path, parts.query, parts.fragment))
helpers.AGENTS[dataplane] = f"{provider_name}'s dataplane"
print(f"Dataplane  {dataplane}")

## The consumer pulls the robots

The provider receives the request on its dataplane, gets a token from the factory's Keycloak, calls the factory and answers with what it returns.

In [ ]:
def latest(robot):
    measures = robot.get("measures") or [{}]
    return max(measures, key=lambda m: m.get("year") or 0)


ROBOT_COLUMNS = [
    ("Robot", "name"),
    ("Model", "model"),
    ("Line", "line"),
    ("Year", lambda r: latest(r).get("year")),
    ("Parts", lambda r: latest(r).get("partsProduced")),
    ("OEE", lambda r: f"{latest(r).get('oee', 0):.0%}"),
    ("Defects", lambda r: f"{latest(r).get('defectRate', 0):.2%}"),
    ("Energy kWh", lambda r: round(latest(r).get("energyKwh", 0))),
    ("Avg °C", lambda r: latest(r).get("avgTemperatureC")),
]

In [ ]:
response = requests.get(dataplane, timeout=90)
robots = show(response, "GET through the dataplane", open_response=False)
table(robots, ROBOT_COLUMNS, title=f"{len(robots)} robots, latest year");

The factory itself refuses anyone without a token. The data reached the consumer only because the provider fetched one: the consumer never saw the factory's credentials.

In [ ]:
factory = env("EUNOMIA_FACTORY_URL")
helpers.AGENTS[factory] = "the factory, directly"
show(requests.get(factory + "/robots/"), "The same robots, straight from the factory, without a token", check=False);

## Suspend, and the dataplane closes (consumer → provider)

In [ ]:
rpc(consumer, "/dsp/current/transfers/rpc/setup-suspension", {**transfer, "code": "SUSPEND", "reason": ["Suspended from the notebook"]}, "Suspension")
show(requests.get(dataplane, timeout=90), "GET through the dataplane, while suspended", check=False);

## Restart, and it opens again (consumer → provider)

In [ ]:
rpc(consumer, "/dsp/current/transfers/rpc/setup-start", transfer, "Restart")
show(requests.get(dataplane, timeout=90), "GET through the dataplane, again", open_response=False);

## The provider completes the transfer (provider → consumer)

After that the dataplane URL is closed for good. A new pull needs a new transfer, on the same agreement.

In [ ]:
rpc(provider, "/dsp/current/transfers/rpc/setup-completion", transfer, "Completion")
show(requests.get(dataplane, timeout=90), "GET through the dataplane, after completion", check=False)
timeline("Everything this notebook asked of the agents");